# From bag-of-words to tf-idf: scaling text features

_Feature Engineering (Zheng & Casari)_

**Weight each word by how rare it is across documents, so common words stop drowning out the signal.**

Start with the bag-of-words matrix $X$: one row per document, one column per word, each entry a raw count. tf-idf is nothing more than scaling each column of that matrix by a single number, the word's inverse document frequency (idf).

       The idf is large for rare words (seen in few documents) and small — near zero — for words seen in nearly every document. Multiply a column of counts by its idf and you stretch the rare, distinctive words while flattening the ubiquitous boilerplate.

---

This notebook is a practice scaffold for the **From bag-of-words to tf-idf: scaling text features** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — scikit-learn

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import (
    CountVectorizer, TfidfTransformer, TfidfVectorizer)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV

# Yelp reviews (Dataset Challenge round 6). Get the data from
# yelp.com/dataset; prepared notebooks: github.com/alicezheng/feature-engineering-book
# A binary task: predict whether a review is for a restaurant (1) or not (0).
review_df = pd.read_json('yelp_academic_dataset_review.json', lines=True)
texts  = review_df['text']
labels = review_df['target']        # 0/1 class column prepared as in the book

train_x, test_x, train_y, test_y = train_test_split(
    texts, labels, train_size=0.7, random_state=123)

# 1) BAG-OF-WORDS: raw counts. Fit the vocabulary on TRAIN only.
bow = CountVectorizer()
X_tr_bow = bow.fit_transform(train_x)
X_te_bow = bow.transform(test_x)            # reuse train vocabulary

# 2) L2-NORMALIZED bag-of-words: scale each document row to unit length.
l2 = TfidfTransformer(norm='l2', use_idf=False)   # norm only, no idf
X_tr_l2 = l2.fit_transform(X_tr_bow)
X_te_l2 = l2.transform(X_te_bow)

# 3) TF-IDF: column-scale the counts by inverse document frequency.
#    idf is FIT ON TRAIN; the test set reuses the train idf.
tfidf = TfidfTransformer()                  # norm='l2', use_idf=True by default
X_tr_tfidf = tfidf.fit_transform(X_tr_bow)
X_te_tfidf = tfidf.transform(X_te_bow)
# (TfidfVectorizer() == CountVectorizer() + TfidfTransformer() in one step.)

def tuned_logreg(Xtr, ytr, Xte, yte):
    """GridSearchCV over the regularization strength C, then test accuracy."""
    grid = {'C': [1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0]}
    search = GridSearchCV(LogisticRegression(max_iter=1000), grid, cv=5)
    search.fit(Xtr, ytr)
    return search.best_params_['C'], search.score(Xte, yte)

for name, Xtr, Xte in [("bag-of-words",       X_tr_bow,   X_te_bow),
                       ("L2-normalized BoW",  X_tr_l2,    X_te_l2),
                       ("tf-idf",             X_tr_tfidf, X_te_tfidf)]:
    best_C, acc = tuned_logreg(Xtr, train_y, Xte, test_y)
    print(f"{name:<20} best C={best_C:<6}  test accuracy={acc:.4f}")

# The book's takeaway: the three accuracies land CLOSE together, and the
# choice of C (regularization) moves the score more than the scaling does.

## Visualize the data & results

_On a tiny real corpus, what happens to a COMMON word's weight versus a RARE word's weight when we switch from raw counts to tf-idf?_

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

# A small REAL inline corpus: 'the' is common across all docs;
# 'volcano' is rare and specific to one doc.
corpus = [
    "the food the the the was good",          # 'the' x4
    "the service was the slow",
    "the place was the busy the",
    "the menu was the long",
    "the volcano roll was the special",       # 'volcano' x1, only here
]

bow = CountVectorizer()
X = bow.fit_transform(corpus).toarray()       # raw bag-of-words counts
vocab = bow.vocabulary_
N = X.shape[0]                                 # number of documents

# The book's plain idf = log(N / n_w), natural log, no smoothing.
# tf-idf is exactly a COLUMN-SCALING of the count matrix by idf.
df  = (X > 0).sum(axis=0)                      # document frequency per word
idf = np.log(N / df)                          # idf(w) = log(N / n_w)
Xt  = X * idf                                  # column-scale -> tf-idf

def weights(word):
    j = vocab[word]
    return X[:, j].max(), Xt[:, j].max()      # raw count, tf-idf weight

the_raw,  the_tfidf  = weights("the")         # 4.0 , 0.0   (idf = log(5/5)=0)
volc_raw, volc_tfidf = weights("volcano")     # 1.0 , 1.609 (idf = log(5/1))
print([round(the_raw,2), round(the_tfidf,2),
       round(volc_raw,2), round(volc_tfidf,2)])
# -> [4.0, 0.0, 1.0, 1.61]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A corpus has $N=1000$ documents. Word "service" appears in 900 of them; word "mediocre" appears in 10. Using the book's plain idf $\log(N/n_w)$ (natural log), which word does tf-idf trust more, and by how much per occurrence?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute idf("service") = log(1000/900). — _Document frequency 900 out of 1000 means the word is nearly ubiquitous, so the ratio is barely above 1._
- Compute idf("mediocre") = log(1000/10). — _Document frequency 10 out of 1000 means the word is rare, so the ratio is 100 and its log is large._
- Compare the two idf values. — _Each occurrence of the word contributes (count × idf), so the idf is the per-occurrence weight._

**Answer:** $\text{idf}(\text{service})=\log(1000/900)=\log(1.111)\approx 0.105$. $\text{idf}(\text{mediocre})=\log(1000/10)=\log(100)\approx 4.605$. tf-idf trusts "mediocre" about $4.605/0.105 \approx 44\times$ more per occurrence — the rare, distinctive word dominates even though "service" is counted far more often across the corpus.

</details>

**Problem 2.** You vectorize your training reviews with tf-idf and get 0.93 cross-validated accuracy. Eager to report test accuracy, you call TfidfVectorizer().fit_transform(test_reviews) and the score jumps. Why is this wrong, and what should you do?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Notice that fit_transform re-learns the idf on the test set. — _The idf weights (and the vocabulary) are now computed from test documents, which the model is supposed to have never seen._
- Recognize this as data leakage. — _Test-set statistics have bled into the features; the inflated score does not reflect real-world performance on unseen text._
- Use the train-fitted vectorizer to only transform the test set. — _The book's rule: idf is fit on train only; test documents reuse the train idf and vocabulary._

**Answer:** It is leakage: fit_transform on the test set recomputes the idf (and vocabulary) from data the model must not see, so the score is optimistic and meaningless. Fit the vectorizer once on the training reviews — vec.fit(train) — then call vec.transform(test) so the test documents reuse the train idf. In a pipeline, put the vectorizer inside the cross-validation so it is refit on each training fold only.

</details>